# 01 · Getting to know M5

We want to use Walmart's M5 data to study forecasting and stocking. Before building a model, I need to understand what one row means, how the files connect, which dates we can evaluate, and where a seemingly reasonable cleaning step might change the answer.

Read the [Technical Glossary and Mathematical Blueprint](../docs/TECHNICAL_BLUEPRINT.md) alongside this notebook. Sales are our observed demand proxy. M5 contains no on-hand stock or lost-sales field.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    (path for path in (working_directory, *working_directory.parents)
     if (path / "configs" / "config.yaml").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Open this notebook from the project root or its notebooks directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_audit import load_config

config_path = project_root / "configs" / "config.yaml"
config = load_config(config_path)
processed = project_root / config["data"]["processed_dir"]

from src.data_audit import run_audit

phases = pd.DataFrame.from_dict(config["splits"], orient="index").rename_axis("phase")
phases["days"] = (
    pd.to_datetime(phases["end"]) - pd.to_datetime(phases["start"])
).dt.days + 1

## 1. What did we actually download?

I started by opening a few rows of each CSV. The three tables describe different things. Sales are in units, the price file has one listed price per store, item and retail week, and the calendar gives real dates to numbered sales columns. I first expected that item ID would connect everything. Before joining, I want to see where it repeats.

The samples below use a few early sales columns and price rows known to precede training. Store counts use identifier columns only. No held-out outcomes are shown.

In [ ]:
raw = project_root / config["data"]["raw_dir"]
source = {kind: raw / name for kind, name in config["data"]["files"].items()}
sales_header = pd.read_csv(source["sales"], nrows=0).columns
identity_fields = ["id", "store_id", "item_id", "dept_id", "cat_id", "state_id"]
catalog = pd.read_csv(source["sales"], usecols=identity_fields)
day_fields = [name for name in sales_header if name.startswith("d_")]
display(pd.read_csv(source["sales"], usecols=[*identity_fields, *day_fields[:5]], nrows=5))
calendar_sample = pd.read_csv(source["calendar"], nrows=5)
display(calendar_sample)
calendar_weeks = pd.read_csv(source["calendar"], usecols=["wm_yr_wk", "date"], parse_dates=["date"])
week_ends = calendar_weeks.groupby("wm_yr_wk")["date"].max()
first_training_day = pd.Timestamp(config["splits"]["training"]["start"])
price_sample = pd.read_csv(source["prices"], nrows=20_000)
historical_prices = price_sample.loc[price_sample["wm_yr_wk"].map(week_ends).lt(first_training_day)]
display(historical_prices.head(5))

stores = catalog.groupby(["state_id", "store_id"]).agg(series=("id", "size"), items=("item_id", "nunique"))
display(stores)
print(f"{len(catalog):,} store–item series across {len(stores)} stores.")
print(f"{catalog['item_id'].nunique():,} distinct item IDs repeat across stores.")
print(f"The wide sales table has {len(day_fields):,} observed day columns, from {day_fields[0]} to {day_fields[-1]}.")

**Scope decision.** Start with CA_1 so every candidate will face the same daily sales and the same simulated stocking conditions. This is a study design chosen in advance. The counts above explain how the audit and feature keys extend to ten stores. They give no reason to expect CA_1's weekday patterns, item mix, or policy rankings to represent the other stores.

### Revisit the proposal's first data questions

The submitted data-understanding notebook drew two hierarchies: state → store, and category → department → item. I can combine both to check that each store contains the same number of series per department. I also try reshaping only three item-day records. That makes it easier to see what converting a wide file to a daily table actually does before applying the full pipeline.

In [ ]:
department_store = pd.crosstab([catalog["cat_id"], catalog["dept_id"]], [catalog["state_id"], catalog["store_id"]])
display(department_store)
preview = pd.read_csv(source["sales"], nrows=1, usecols=["store_id", "item_id", *day_fields[:3]])
tiny_daily = preview.melt(id_vars=["store_id", "item_id"], var_name="d", value_name="recorded units")
display(tiny_daily.merge(calendar_sample[["d", "date"]], on="d", validate="many_to_one"))

### What should we join on?

| Source | One row represents | Join fields |
|---|---|---|
| Sales | One store and item, with a column per observed day | Store, item and M5 day after reshaping |
| Calendar | One day, with events and state-specific SNAP flags | M5 day and retail week |
| Prices | Listed price for one store, item and retail week | Store, item and retail week |

Ingestion checks configured hashes, schema, key uniqueness and join cardinality. The daily left join preserves sales rows that lack a listed weekly price. Missing prices remain null with an explicit flag. A gap may mark a pre-launch or inactive week. M5 cannot establish the reason for each gap.

The audit checks coverage across all configured dates. Examples and price analysis below use training dates.

In [ ]:
# Try the shortcut on a single historical observation, safely away from the full panel.
example_prices = historical_prices
example_keys = ["store_id", "item_id", "wm_yr_wk"]
example = example_prices.loc[:, example_keys].iloc[[0]]
example_item = example["item_id"].iloc[0]
example_store = example["store_id"].iloc[0]
# The raw catalog establishes duplication across stores. The price sample establishes it across weeks.
item_prices = example_prices.loc[example_prices["item_id"].eq(example_item)]
display(pd.DataFrame([
    {"attempt": "Identify a sales series by item alone", "matches": int(catalog["item_id"].eq(example_item).sum())},
    {"attempt": "Find a price by store and item (sample)", "matches": int(item_prices["store_id"].eq(example_store).sum())},
    {"attempt": "Find a price by store, item and week (sample)", "matches": len(example.merge(item_prices, on=example_keys))},
]))

The test explains why the full audit checks the grain of each table and verifies join cardinality. Using only store and item to attach prices would repeat one day's sales for every matched week and inflate totals. The sample explains the failure mode. The reusable audit applies the actual checks to the full files.

In [ ]:
audit = run_audit(config_path)
sources = pd.DataFrame.from_dict(audit["sources"], orient="index").rename_axis("source")
display(sources[["filename", "bytes", "sha256"]])
display(pd.DataFrame([
    {"store": store, "series": report["series"], "item-days": report["summary"]["rows"]}
    for store, report in audit["stores"].items()
]))

## 2. How far can we actually evaluate?

The calendar continues after the last day with recorded sales. I can use a future date's scheduled calendar fields at a historical forecast origin under our availability assumption. If I fill later sales with zero, I create artificial targets. The audit keeps calendar dates and observed sales coverage distinct.

Phase boundaries are an experimental choice. We leave history before training for lags, tune forecast design on development, use a separate calibration period for forecast uncertainty, and reserve a later block for picking inventory policy settings. Warm-up gives stock and open orders time to accumulate before evaluation. Choosing a random split would let later outcomes influence earlier experimental decisions.

In [ ]:
display(pd.DataFrame([
    {"source": "observed sales", "start": audit["observed_sales"]["start"], "end": audit["observed_sales"]["end"]},
    {"source": "calendar", "start": audit["calendar"]["start"], "end": audit["calendar"]["end"]},
    {"source": "prepared panel", "start": audit["panel"]["start"], "end": audit["panel"]["end"]},
]))
horizons = config["forecast"]["cumulative_horizons"]
window_counts = pd.DataFrame([
    {
        "phase": name,
        "dates with observed targets": phase["target_days"],
        **{f"complete {h}-day windows": phase["complete_windows_by_horizon"][str(h)] for h in horizons},
    }
    for name, phase in audit["splits"].items()
]).set_index("phase")
display(window_counts)

display(phases)

### What does a missing calendar event mean?

The original notebook counted both event slots by year. A blank slot normally means no named event in that slot. It is different from a missing sale or a missing weekly price. Some dates have two events, so I count dates separately from event slots. The first and last calendar years have fewer dates and deserve care when comparing counts.

In [ ]:
schedule = pd.read_csv(source["calendar"], usecols=["year", "event_type_1", "event_type_2"])
summary = pd.DataFrame({
    "calendar dates": schedule.groupby("year").size(),
    "dates with any named event type": schedule[["event_type_1", "event_type_2"]].notna().any(axis=1).groupby(schedule["year"]).sum(),
    "dates with both slots filled": schedule[["event_type_1", "event_type_2"]].notna().all(axis=1).groupby(schedule["year"]).sum(),
})
event_slots = schedule.melt(id_vars="year", value_vars=["event_type_1", "event_type_2"], value_name="event type")
display(summary)
display(pd.crosstab(event_slots["event type"], event_slots["year"]))

**Consequence for the experiment.** Neighboring origins share target days, so the number of windows overstates the number of independent observations. Pair the same valid origins when comparing two-day and eight-day forecast scores. The short April test can expose a large failure. It gives limited evidence that a small advantage would persist across months or stores.

### First check the joined panels

The table separates two different observations, namely a zero in the sales CSV and an absent record in the prices CSV. Both can occur on the same row. The final assertion checks that the flag agrees with the actual null values.

In [ ]:
training = config["splits"]["training"]
start, end = pd.Timestamp(training["start"]), pd.Timestamp(training["end"])
key = ["store_id", "item_id", "date"]
checks, examples = [], []

for store in config["data"]["stores"]:
    panel = pd.read_parquet(
        processed / "panel" / f"{store}.parquet",
        columns=[*key, "sales", "sell_price", "price_missing"],
    )
    assert len(panel) == audit["panel"]["days"] * config["data"]["expected_series_per_store"]
    assert not panel.duplicated(key).any()
    assert panel["date"].nunique() == audit["panel"]["days"]
    assert (panel["price_missing"] == panel["sell_price"].isna()).all()
    train = panel.loc[panel["date"].between(start, end)]
    for label, mask in [
        ("Listed weekly price", ~train["price_missing"]),
        ("Missing weekly price", train["price_missing"]),
    ]:
        sales = train.loc[mask, "sales"]
        checks.append({
            "store": store,
            "price status": label,
            "zero sales item-days": int(sales.eq(0).sum()),
            "positive sales item-days": int(sales.gt(0).sum()),
        })
    examples.append(train.head(3))
    del panel, train

display(pd.DataFrame(checks))
display(pd.concat(examples, ignore_index=True))

## 3. Would dropping missing prices fix the dataset?

The table above tells us whether price gaps and recorded zeros show up together. Next I want to know where the gaps sit in each item's timeline. A zero before the first observed sale is different evidence from a zero after sales have already been recorded. An item with no positive days during training needs its own row in the diagnosis.

An inner join would discard missing-price days. Compare how much it would reduce the number of item-days and how it would change the observed zero share. This is a small counterfactual about data preparation and uses training dates only.

In [ ]:
focus_store = config["data"]["stores"][0]
history = pd.read_parquet(
    processed / "panel" / f"{focus_store}.parquet",
    columns=["item_id", "date", "sales", "sell_price", "price_missing", "week_end"],
    filters=[("date", ">=", start), ("date", "<=", end)],
)
first_sale = history.loc[history["sales"].gt(0)].groupby("item_id")["date"].min()
first_sale_for_row = history["item_id"].map(first_sale)
position = pd.Series("On/after first recorded training sale", index=history.index)
position.loc[history["date"].lt(first_sale_for_row)] = "Before first recorded training sale"
position.loc[first_sale_for_row.isna()] = "No positive day in training"
timeline_counts = (
    history.assign(position=position, zero_sales=history["sales"].eq(0))
    .groupby("position")
    .agg(
        item_days=("sales", "size"),
        zero_share=("zero_sales", "mean"),
        missing_price_days=("price_missing", "sum"),
        missing_price_share=("price_missing", "mean"),
    )
)
display(timeline_counts.round(3))

join_effect = pd.DataFrame([
    {
        "preparation": label,
        "item-days": len(sample),
        "units": int(sample["sales"].sum()),
        "zero share": sample["sales"].eq(0).mean(),
    }
    for label, sample in [
        ("Keep calendar and all sales days (left join)", history),
        ("Keep days with listed price only (inner join)", history.loc[~history["price_missing"]]),
    ]
]).set_index("preparation")
display(join_effect.round(3))
left, inner = join_effect.iloc[0], join_effect.iloc[1]
print(f"Dropping missing-price days removes {int(left['item-days'] - inner['item-days']):,} item-days "
      f"and {int(left['units'] - inner['units']):,} recorded units. "
      f"The zero share changes from {left['zero share']:.2%} to {inner['zero share']:.2%}.")

**Cleaning decision.** Keep the complete sales calendar, preserve null prices, and record a price-missing flag. Dropping rows changes how sparse a series looks and can break fixed daily lags. Replacing a missing price with zero would manufacture a free item. Backfilling can import a later price into an earlier day. Zero sales can also occur while a price is listed. We cannot observe whether those are quiet days, stockouts, or another circumstance.

### Would we have known the joined price at the origin?

The weekly file does not record its publication time. The table below separates daily rows in a week that has already ended as of the observation date from rows carrying a price for a week still in progress on that date. This comparison uses the project's conservative completed-week rule. A retailer with timestamped, pre-announced prices could use a less conservative rule supported by its records.

In [ ]:
priced_days = history.loc[history["sell_price"].notna()]
week_status = priced_days["date"].lt(priced_days["week_end"]).map({
    True: "Joined week's end is still in the future",
    False: "Joined week ends on the origin date",
})
display(week_status.value_counts().rename("priced item-days").to_frame())
weeks = pd.read_parquet(
    processed / "price_history" / f"{focus_store}.parquet",
    filters=[("week_end", "<=", end)],
)
example_item = history.loc[~history["price_missing"], "item_id"].iloc[0]
available = weeks.loc[weeks["item_id"].eq(example_item), ["week_end", "sell_price"]].sort_values("week_end")
sample_day = history.loc[history["item_id"].eq(example_item) & ~history["price_missing"]].iloc[0]
at_origin = available.loc[available["week_end"].le(sample_day["date"])].tail(1)
print(f"Example origin {sample_day['date'].date()} for {example_item}. "
      f"The panel attaches week ending {sample_day['week_end'].date()}.")
display(at_origin.rename(columns={"week_end": "eligible price week end", "sell_price": "as-of price"}))

## 4. What this leaves unresolved

| Question | What we can use | What would resolve the uncertainty |
|---|---|---|
| Was a zero a day without demand? | Recorded units and price presence as context | Stock-on-hand, stockout flags, open hours and lost-sales records |
| Did an item launch or delist here? | First and last signs of trading as exploratory markers | Store-level assortment effective dates |
| Did a listed price arrive before a buyer ordered? | Completed retail week as a reproducible availability proxy | Timestamped price publication or planned price snapshots |
| Does CA_1 generalize? | Compatible keys across the ten stores | A separate replication on other stores under the same rules |

These distinctions carry into Notebook 02. We can compare how alternative definitions change the data we analyze, while keeping the main protocol and test dates fixed.

### Files we will use next

The processed folder contains one daily panel and one price-history file per store, the daily calendar, and the JSON audit. Price history records the week end so later feature code can apply the availability rule. The joined same-week price in the daily panel serves audit and exploration.

Open [Notebook 02](02_eda_and_demand_regimes.ipynb) for training-only demand profiles.